# Sources

What do the sources have? All their extracts can be listed, also as a GeoDataFrame.

In [ ]:
%pip install osmfinder basemaplibre geopandas matplotlib

In [ ]:
import basemaplibre as bml
import matplotlib.pyplot as plt
import pandas as pd

import osmfinder
from osmfinder.extract import OsmExtractsIndex

In [ ]:
sources = [source for source in osmfinder.OsmExtractSource if source != "any"]
extracts = pd.concat(
    OsmExtractsIndex.from_extracts(osmfinder.get_available_extracts(source))
    .to_geodataframe()
    .assign(source=source.value)
    for source in sources
)
extracts.sample(5, random_state=0)[["source", "id", "name", "file_name", "area"]]

## Number and size

In [ ]:
extracts.groupby("source")["area"].describe().round().astype(int)[
    ["count", "min", "25%", "50%", "75%", "max"]
]

## Coverage

BBBike has cities, Movisda a grid of one degree, the other sources administrative regions: continents with countries in them, and some of the countries with their regions. The darker the place, the more extracts there are to choose from:

In [ ]:
europe = extracts.cx[-11:32, 35:62].to_crs(3035)

fig, axs = plt.subplots(4, 2, figsize=(12, 21), dpi=100, layout="constrained")
axs.flat[-1].set_axis_off()
for ax, (source, group) in zip(axs.flat, europe.groupby("source"), strict=False):
    ax.set_axis_off()
    ax.axis((2_500_000, 6_000_000, 1_400_000, 4_700_000))
    group[group["area"] < 5e6].plot(ax=ax, alpha=0.2, edgecolor="steelblue", linewidth=0.5)
    ax.axis((2_500_000, 6_000_000, 1_400_000, 4_700_000))
    bml.add_basemap(ax, crs=3035, source="OpenFreeMap.Positron", labels=False, attribution_size=6)
    ax.set_title(f"{source}: {len(group)} extracts")
plt.show()

## Tree

`display_available_extracts` prints the extracts of a source as they are nested. `head` limits how many of them:

In [ ]:
osmfinder.display_available_extracts("Geofabrik", head=15)